## Global instructions

Only three input files are required (provided with example data):
1) AAseq.csv
2) overhangs.csv
3) orthogonal_oligos.csv

AAseq.csv should be your variant names and their associated amino acid sequences

overhangs.csv defines the set of high-fiedlity Golden-gate overhangs to use. I use NEB GETset (https://ligasefidelity.neb.com/getset/run.cgi) and enter my required vector overhangs (GCTT and AGTG) to retrieve overhang sets compatible with those. I typically use 24 but more or less can also be used. 

orthogonal_oligos.csv contains the set or primers to use for amplification in 5'-3' orientation and will be used to add those binding sites to each sub-pool uniquely


In Gene Optimisation certain variables can be defined e.g. restriction sites to avoid, homopolymers to avoid, GC content, host to optimise for etc. all defined by the DNAchisel tool (https://edinburgh-genome-foundry.github.io/DnaChisel/index.html). This step can also be skipped if optimised elsewhere and supplied to Pool Assignment in the file Optimised_genes.csv

In Pool Assignment certain variables can be defined e.g. length of oPool (I typically use 300 or 250nt), the defined vector overhangs (to check they are not in your overhang list to assign to pools) and the maximum size of a 'short' sub-pool i.e. genes requiring only one fragment for assembly. 

In Primer and BsaI site addition certain variable can be defined e.g. Vector overhangs, the restriction enzyme site (including an A,G,C or T for its N e.g. BsaI is GGTCTCN| so I pass it GGTCTCA"). This will output three files, one FULL_INFO.csv with all information on pool assignment, primers to use, number of fragments etc., one references.fasta containing all gene sequences, and one oPool_Order_Fragments.csv which can be uploaded directly to Twist for ordering! 




## Gene optimisation

### Variables

In [ ]:
pip install dnachisel


In [ ]:
import pandas as pd
from Bio.Seq import Seq
from Bio.Data import CodonTable
from dnachisel import (
    DnaOptimizationProblem,
    AvoidPattern,
    EnforceGCContent,
    EnforceTranslation,
    CodonOptimize,
)
# ─── User‐configurable parameters ───
# List of short sequences (e.g. restriction sites or other motifs) to avoid. Crucially include the restriction enzyme you will use for Golden-gate!
AVOID_SEQS = [
    "GGTCTC",      # BsaI
    "AAAAA",      # Homopolymers
    "GGGGG", 
    "CCCCC",
    "TTTTT"     # hypothetical extra pattern
    # …add as many as you like…
]
# GC‐content constraints:
GC_MIN    = 0.30
GC_MAX    = 0.70
GC_WINDOW = 50

# Codon‐optimization settings:
CODON_SPECIES = "e_coli"
CODON_METHOD  = "match_codon_usage"

input_path = "AAseq.csv"

output_path = "Optimised_Genes.csv"

input_df = pd.read_csv(input_path, names=["name", "aa_seq"])

input_df

### Script

#### Reverse translates amino acid sequences into DNA

In [ ]:
def reverse_translate(protein_sequence: str) -> str:
    """
    Reverse translate an amino acid sequence to a DNA sequence using the standard codon table.
    Chooses the first available codon for each amino acid.

    Args:
        protein_sequence (str): Amino acid sequence using 1-letter codes.

    Returns:
        str: A DNA sequence (string of A, T, G, C).
    """
    # Get the standard codon table
    codon_table = CodonTable.unambiguous_dna_by_id[1]

    # Create a dictionary mapping amino acids to one possible codon
    aa_to_codon = {}
    for codon, aa in codon_table.forward_table.items():
        if aa not in aa_to_codon:
            aa_to_codon[aa] = codon.upper()

    # Add stop codon
    aa_to_codon['*'] = codon_table.stop_codons[0].upper()

    # Translate the amino acid sequence
    dna_sequence = ''
    for aa in protein_sequence.upper():
        if aa not in aa_to_codon:
            raise ValueError(f"Invalid amino acid: {aa}")
        dna_sequence += aa_to_codon[aa]

    return dna_sequence


input_df["dna_seq_unrefined"] = input_df.aa_seq.apply(lambda x: reverse_translate(x))

#### Optimises DNA sequences

In [ ]:
def dna_chisel_remove_RS_and_codon_optimize(
    dna_seq: str,
    avoid_seqs: list[str] | None = None
) -> str:
    """
    Run DNA Chisel on `dna_seq`, removing any short motifs in `avoid_seqs`
    and then codon‐optimizing the entire open reading frame.

    If `avoid_seqs` is None, it uses the global AVOID_SEQS defined in Cell 1.
    """

    # Use the passed-in list or fall back to the global list
    patterns_to_avoid = avoid_seqs if (avoid_seqs is not None) else AVOID_SEQS

    seq_length = len(dna_seq)

    # Build one AvoidPattern constraint per motif in patterns_to_avoid
    avoid_constraints = [AvoidPattern(pat) for pat in patterns_to_avoid]

    problem = DnaOptimizationProblem(
        sequence=dna_seq,
        constraints=[
            *avoid_constraints,
            EnforceGCContent(mini=GC_MIN, maxi=GC_MAX, window=GC_WINDOW),
            EnforceTranslation(location=(0, seq_length)),
        ],
        objectives=[
            CodonOptimize(
                species=CODON_SPECIES,
                method=CODON_METHOD,
                location=(0, seq_length),
            )
        ]
    )

    # Solve and optimize
    problem.resolve_constraints()
    problem.optimize()

    return problem.sequence

input_df["dna_seq_optimized"] = input_df.dna_seq_unrefined.apply(lambda x: dna_chisel_remove_RS_and_codon_optimize(x))

input_df = input_df.drop(columns=["dna_seq_unrefined"])

input_df.to_csv(output_path, index=False)


## Pool Assignment

### Variables and imports

In [ ]:
import os
import itertools
import pandas as pd
import networkx as nx
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.Data import CodonTable

# ----------------------------
# PARAMETERS / USER INPUT
# ----------------------------

# 1) oPool length options: 300, 250, 200, 150, 100
OPOOL_LENGTH = 250  # ← Change to one of [300, 250, 200, 150, 100]

# 2) Fixed vector overhangs (exactly two, 4-nt each)
VECTOR_OVERHANG_1 = "GCTT"
VECTOR_OVERHANG_2 = "AGTG"
# These must NOT appear in the OVERHANGS_FILE.

# 3) SHORT-pool limit - for one fragment assemblies only which CAN have unlimited pool size or a defined max if desired
SHORT_POOL_MAX_SIZE = 50  # integer or None for “unlimited”

# 4) Filepaths
OVERHANGS_FILE = "overhangs.csv"
INPUT_FILE     = "Optimised_Genes.csv"
OUTPUT_FILE    = "Assigned_Genes.csv"
UNASSIGNED_LOG = "unassigned_genes.csv"

### Script

In [ ]:
# ----------------------------
# GLOBAL SETUP (once)
# ----------------------------
def load_overhangs(path):
    """
    Reads a CSV whose first cell is a comma-separated list of 4-nt overhangs.
    Returns a list of 4-nt strings (uppercase).
    """
    df = pd.read_csv(path, header=None)
    raw = str(df.iloc[0, 0])
    return [oh.strip().upper() for oh in raw.split(",") if oh.strip()]

OVERHANGS_LIST = load_overhangs(OVERHANGS_FILE)

# Build codon table & synonyms
codon_table = CodonTable.unambiguous_dna_by_name["Standard"]
syn_map = {}
for codon, aa in codon_table.forward_table.items():
    syn_map.setdefault(aa, []).append(codon)


# ----------------------------
# HELPER FUNCTIONS
# ----------------------------
def load_sequences(path):
    """
    Read a CSV with headers [name, aa_seq, dna_seq_optimized].
    Returns a list of SeqRecord, using 'name' as the record ID and
    'dna_seq_optimized' as the sequence.
    """
    df = pd.read_csv(path)
    records = []
    for _, row in df.iterrows():
        gid = str(row["name"]).strip()
        seq = str(row["dna_seq_optimized"]).strip().upper()
        if gid and seq:
            records.append(SeqRecord(Seq(seq), id=gid, description=""))
    return records


def try_synonymous_substitution(seq_str, codon_start):
    """
    Yield all candidate sequences obtained by replacing the codon at codon_start
    with each synonymous codon. If original codon is not a standard AA codon, yield nothing.
    """
    old_codon = seq_str[codon_start:codon_start + 3]
    aa = codon_table.forward_table.get(old_codon)
    if aa is None:
        return
    for syn_codon in syn_map.get(aa, []):
        if syn_codon == old_codon:
            continue
        yield seq_str[:codon_start] + syn_codon + seq_str[codon_start + 3:]


def enforce_overhang_at_pos(seq_str, overhang, start_pos):
    """
    Ensure seq_str[start_pos:start_pos+4] == overhang by ≤ one synonymous swap.
    Return (modified_seq, type_flag), where type_flag in {"Native","Synonymous"}.
    If impossible, return (None, None).
    """
    if seq_str[start_pos:start_pos + 4] == overhang:
        return seq_str, "Native"
    for offset in range(4):
        idx = start_pos + offset
        codon_start = idx - (idx % 3)
        if codon_start < 0 or codon_start + 3 > len(seq_str):
            continue
        for candidate in try_synonymous_substitution(seq_str, codon_start):
            if candidate[start_pos:start_pos + 4] == overhang:
                return candidate, "Synonymous"
    return None, None


def get_pos_to_options(seq_str):
    """
    Build a dict mapping each cut position → list of (overhang, type_flag),
    where type_flag is "Native" or "Synonymous," for every overhang in OVERHANGS_LIST
    that can appear (by at most one synonymous swap) at seq_str[cut_pos-4:cut_pos].
    """
    L = len(seq_str)
    pos_to_options = {}
    codon_cache = {}

    for cut in range(4, L + 1):
        window = seq_str[cut - 4:cut]
        candidates = []

        # 1) Native window
        if window in OVERHANGS_LIST:
            candidates.append((window, "Native"))

        # 2) Single‐codon synonyms
        for i in range(cut - 4, cut):
            codon_start = i - (i % 3)
            if codon_start < 0 or codon_start + 3 > L:
                continue
            if codon_start not in codon_cache:
                variants = []
                for seq_mut in try_synonymous_substitution(seq_str, codon_start):
                    variants.append(seq_mut)
                codon_cache[codon_start] = variants
            for seq_mut in codon_cache[codon_start]:
                mut_window = seq_mut[cut - 4:cut]
                if mut_window in OVERHANGS_LIST:
                    candidates.append((mut_window, "Synonymous"))

        if candidates:
            seen = set()
            unique_list = []
            for oh, tflag in candidates:
                key = (oh, tflag)
                if key not in seen:
                    seen.add(key)
                    unique_list.append((oh, tflag))
            pos_to_options[cut] = unique_list

    return pos_to_options


def find_valid_n_oh_combo(gene_id, info, pool_free_ohs, n):
    """
    Given:
      - info: {'seq': sequence, 'length': L, 'pos_to_opts': {cut_pos: [(oh,type), …], …}}
      - pool_free_ohs: set of overhangs still available in this pool
      - n: number of fragments desired (so we need n-1 overhangs)
    Attempt to find ANY tuple (chosen_ohs, seq_mod, cuts) where:
      • chosen_ohs is a list of n-1 distinct overhangs ∈
        (overhangs appearing in info['pos_to_opts']) ∩ pool_free_ohs
      • cuts is an ordered list [cut1, cut2, …, cut_{n-1}]
      • seq_mod is the sequence after enforcing each overhang in order
      • Each fragment i (from prev_cut to cut_i) satisfies the oPool fragment‐length constraint
    If found, return (chosen_ohs, seq_mod, cuts). Otherwise return (None, None, None).
    """
    seq_str = info['seq']
    L = info['length']
    pos_to_opts = info['pos_to_opts']
    overhangs_needed = n - 1

    # 1) Build the set of overhangs that appear in pos_to_opts and are still free in the pool
    available_ohs = {oh for opts in pos_to_opts.values() for (oh, _) in opts}
    candidates = list(available_ohs & pool_free_ohs)
    if len(candidates) < overhangs_needed:
        return (None, None, None)

    # 2) Try every ordered combination of size (n-1)
    for oh_combo in itertools.permutations(candidates, overhangs_needed):
        seq_curr = seq_str
        prev_cut = 0
        cuts = []
        success = True

        for i, oh in enumerate(oh_combo):
            # Determine this fragment’s max length
            if i < (n - 1):
                frag_max = OPOOL_LENGTH - 58
            else:
                frag_max = OPOOL_LENGTH - 62
            # Compute sum of remaining fragments’ maxima minus the necessary 4nt per overhang
            remaining_max_sum = sum(
                [(OPOOL_LENGTH - 58)] * ((n - 1) - i - 1)
                + [(OPOOL_LENGTH - 62)]
            ) - 4 * ((n - 1) - i - 1)

            possible_cuts = sorted(
                [cut for cut, opts in pos_to_opts.items() if any(o == oh for (o, _) in opts)]
            )

            placed = False
            for cut in possible_cuts:
                if cut <= prev_cut:
                    continue
                frag_len = cut - prev_cut
                if frag_len > frag_max:
                    break
                rem_coding = L - (cut - 4)
                if rem_coding > remaining_max_sum:
                    continue

                seq_after, _ = enforce_overhang_at_pos(seq_curr, oh, cut - 4)
                if seq_after is None:
                    continue

                cuts.append(cut)
                seq_curr = seq_after
                prev_cut = cut
                placed = True
                break

            if not placed:
                success = False
                break

        if success:
            return (list(oh_combo), seq_curr, cuts)

    return (None, None, None)


def enhanced_assign_n_part_phase(genes, codon_table, syn_map, n, total_overhangs):
    """
    PHASE n≥3: Assign all `genes` via n-part assembly, using the optimized
    pos_to_options logic + find_valid_n_oh_combo for each pool.
    Returns:
      - surviving_pools: list of {'records': [(gid, ohs, cuts, seq_mod),…], 'free_ohs': set(...) }
      - next_phase: list of SeqRecords that either failed direct or were too long
      - pool_sizes: list of sizes for each surviving pool
    """

    gene_info = {}
    too_long = []
    candidates = []
    max_bases = (n - 1) * (OPOOL_LENGTH - 58) + (OPOOL_LENGTH - 62) - 4 * (n - 1)

    # 1) Precompute pos_to_options for each gene, flag too_long or zero-options
    for rec in genes:
        seq_str = str(rec.seq)
        L = len(seq_str)
        if L > max_bases:
            too_long.append(rec)
        else:
            pos_to_opts = get_pos_to_options(seq_str)
            if not pos_to_opts:
                candidates.append((rec, None))
            else:
                gene_info[rec.id] = {
                    'seq': seq_str,
                    'length': L,
                    'pos_to_opts': pos_to_opts
                }
                candidates.append((rec, gene_info[rec.id]))

    failed_direct = []
    actual_candidates = []
    for rec, info in candidates:
        if info is None:
            failed_direct.append(rec)
        else:
            actual_candidates.append(rec)

    # 2) Sort actual_candidates by (#options, -length)
    gene_options = {}
    for rec in actual_candidates:
        info = gene_info[rec.id]
        tot = sum(len(opts) for opts in info['pos_to_opts'].values())
        gene_options[rec.id] = tot

    actual_candidates.sort(key=lambda r: (gene_options[r.id], -len(r.seq)))

    # 3) Build global frequency for overhangs
    global_gene_count = {oh: 0 for oh in OVERHANGS_LIST}
    for rec in actual_candidates:
        info = gene_info[rec.id]
        for opts in info['pos_to_opts'].values():
            for oh, _ in opts:
                global_gene_count[oh] += 1

    pools = []
    max_per_pool = total_overhangs // (n - 1)
    blocking = actual_candidates[:]

    # 4) Attempt to place each candidate into existing pools (or open new)
    while blocking:
        next_block = []
        for rec in blocking:
            info = gene_info[rec.id]
            placed = False

            # Try each existing pool
            for pool in pools:
                if len(pool['records']) < max_per_pool and len(pool['free_ohs']) >= (n - 1):
                    ohs, seq_mod, cuts = find_valid_n_oh_combo(
                        rec.id, info, pool['free_ohs'], n
                    )
                    if ohs is not None:
                        pool['records'].append((rec.id, ohs, cuts, seq_mod))
                        for oh in ohs:
                            pool['free_ohs'].remove(oh)
                        placed = True
                        break

            if not placed:
                next_block.append(rec)

        # If none placed, open a new pool or mark failed_direct
        if len(next_block) == len(blocking):
            rec0 = next_block.pop(0)
            info0 = gene_info.get(rec0.id, None)
            if info0 is None:
                failed_direct.append(rec0)
            else:
                ohs, seq_mod, cuts = find_valid_n_oh_combo(
                    rec0.id, info0, set(OVERHANGS_LIST), n
                )
                if ohs is not None:
                    pools.append({
                        'records': [(rec0.id, ohs, cuts, seq_mod)],
                        'free_ohs': set(OVERHANGS_LIST) - set(ohs)
                    })
                else:
                    failed_direct.append(rec0)

        blocking = next_block

    # 5) Cull pools smaller than threshold (except keep singletons)
    surviving = []
    culled_ids = []
    threshold = (total_overhangs // (n - 1)) // 3

    for pool in pools:
        size = len(pool['records'])
        if size >= threshold or size == 1:
            surviving.append(pool)
        else:
            for (gid, _, _, _) in pool['records']:
                culled_ids.append(gid)

    culled_remain = [r for r in genes if r.id in culled_ids]
    next_phase = failed_direct + too_long + culled_remain
    pool_sizes = [len(p['records']) for p in surviving]
    return surviving, next_phase, pool_sizes


def assign_short_phase(short_recs, start_block):
    """
    Put all short sequences (≤ OPOOL-62) into one or more short pools,
    splitting into blocks of up to SHORT_POOL_MAX_SIZE if not None.
    Returns a list of dict rows for CSV output.
    """
    rows = []
    if SHORT_POOL_MAX_SIZE is None:
        for rec in short_recs:
            rows.append({
                "Block": start_block,
                "Length Distribution": "Short",
                "Sequence Name": rec.id,
                "Overhang1": "N/A",
                "Overhang2": "N/A",
                "DNA Fragment 1": str(rec.seq),
                "DNA Fragment 2": "",
                "DNA Fragment 3": "",
                "Full Sequence": str(rec.seq)
            })
    else:
        block_idx = start_block
        for i in range(0, len(short_recs), SHORT_POOL_MAX_SIZE):
            for rec in short_recs[i:i + SHORT_POOL_MAX_SIZE]:
                rows.append({
                    "Block": block_idx,
                    "Length Distribution": "Short",
                    "Sequence Name": rec.id,
                    "Overhang1": "N/A",
                    "Overhang2": "N/A",
                    "DNA Fragment 1": str(rec.seq),
                    "DNA Fragment 2": "",
                    "DNA Fragment 3": "",
                    "Full Sequence": str(rec.seq)
                })
            block_idx += 1
    return rows


def write_unassigned_log(unassigned_ids, csv_path):
    df = pd.DataFrame({"Sequence Name": unassigned_ids})
    df.to_csv(csv_path, index=False)


# ----------------------------
# MAIN PIPELINE
# ----------------------------
def main_pipeline():
    # 1) Verify vector overhangs
    vec_ohs = {VECTOR_OVERHANG_1.upper(), VECTOR_OVERHANG_2.upper()}
    if len(vec_ohs) != 2:
        raise ValueError("VECTOR_OVERHANG_1 and VECTOR_OVERHANG_2 must be two distinct 4-nt strings.")
    for oh in OVERHANGS_LIST:
        if oh.upper() in vec_ohs:
            raise ValueError(f"Overhang file contains vector overhang {oh}.")
    total_oh = len(OVERHANGS_LIST)

    # 2) Load input sequences
    recs = load_sequences(INPUT_FILE)

    # 3) Split into short vs multi‐part candidates
    short_list = []
    multi_list = []
    for rec in recs:
        if len(rec.seq) <= (OPOOL_LENGTH - 62):
            short_list.append(rec)
        else:
            multi_list.append(rec)

    rows = []
    if SHORT_POOL_MAX_SIZE is None:
        rows.extend(assign_short_phase(short_list, start_block=1))
        current_block = 2
    else:
        short_rows = assign_short_phase(short_list, start_block=1)
        rows.extend(short_rows)
        current_block = max(r["Block"] for r in short_rows) + 1 if short_rows else 1

    unassigned = []

    # 5) Phase 2: TWO‐PART, then immediate pass of zero‐option/unmatched into n=3
    MAX_TWO_1 = OPOOL_LENGTH - 58
    MAX_TWO_2 = OPOOL_LENGTH - 62
    current_genes = multi_list

    two_part_candidates = []
    no_two_part = []
    for rec in current_genes:
        seq_str = str(rec.seq)
        L = len(seq_str)
        pos_to_opts = get_pos_to_options(seq_str)
        valid = {}
        for cut, opts in pos_to_opts.items():
            frag1_len = cut
            frag2_len = L - (cut - 4)
            if frag1_len <= MAX_TWO_1 and frag2_len <= MAX_TWO_2:
                valid[cut] = opts
        if valid:
            two_part_candidates.append((rec, valid))
        else:
            no_two_part.append(rec)

    # Build gene_to_validCuts for matching
    gene_to_validCuts = {}
    for rec, valid in two_part_candidates:
        tmp = {}
        for cut, opts in valid.items():
            for oh, _ in opts:
                tmp.setdefault(oh, []).append(cut)
        if tmp:
            gene_to_validCuts[rec.id] = tmp

    def can_assign_two_part_inner(g2v):
        """
        Binary‐search + Hopcroft–Karp on g2v = gene → {oh → [cuts]}.
        Returns (assignment_dict, T) or (None,None).
        """
        genes_2 = list(g2v.keys())
        if not genes_2:
            return None, None
        unique_ohs = sorted({oh for d in g2v.values() for oh in d})

        def feasible(T):
            G = nx.Graph()
            G.add_nodes_from(genes_2, bipartite=0)
            right_nodes = []
            for oh in unique_ohs:
                for i in range(T):
                    right_nodes.append(f"{oh}_{i}")
            G.add_nodes_from(right_nodes, bipartite=1)
            for gene, valid_dict in g2v.items():
                for oh in valid_dict:
                    for i in range(T):
                        G.add_edge(gene, f"{oh}_{i}")
            matching = nx.algorithms.bipartite.matching.hopcroft_karp_matching(G, top_nodes=genes_2)
            matched = sum(1 for g in genes_2 if g in matching)
            return matching if matched == len(genes_2) else None

        low, high = 1, len(genes_2)
        best_T = None
        best_matching = None
        while low <= high:
            mid = (low + high) // 2
            m = feasible(mid)
            if m:
                best_T = mid
                best_matching = m.copy()
                high = mid - 1
            else:
                low = mid + 1

        if best_matching is None:
            return None, None

        assignment = {}
        for gene in genes_2:
            clone_node = best_matching[gene]
            oh, idx_str = clone_node.rsplit("_", 1)
            pool_idx = int(idx_str) + 1
            cut_pos = g2v[gene][oh][0]
            assignment[gene] = (oh, pool_idx, cut_pos)
        return assignment, best_T

    assignment_2, T2 = can_assign_two_part_inner(gene_to_validCuts)
    remaining_after_two = []

    if assignment_2:
        for gid, (oh, pool_idx, cut_pos) in assignment_2.items():
            seq_str = next(rec.seq for rec, _ in two_part_candidates if rec.id == gid)
            seq_mod, _ = enforce_overhang_at_pos(seq_str, oh, cut_pos - 4)
            frag1 = seq_mod[:cut_pos]
            frag2 = seq_mod[cut_pos - 4:]
            rows.append({
                "Block": pool_idx + (current_block - 1),
                "Length Distribution": "Long-2part",
                "Sequence Name": gid,
                "Overhang1": oh,
                "Overhang2": "N/A",
                "DNA Fragment 1": frag1,
                "DNA Fragment 2": frag2,
                "DNA Fragment 3": "",
                "Full Sequence": seq_mod
            })
        matched_ids = set(assignment_2.keys())
        for rec, valid in two_part_candidates:
            if rec.id not in matched_ids:
                remaining_after_two.append(rec)
    else:
        for rec, valid in two_part_candidates:
            remaining_after_two.append(rec)

    current_genes = no_two_part + remaining_after_two
    current_block += (T2 or 0)

    # 6) Phase n≥3
    n = 3
    while current_genes:
        overhangs_needed = n - 1
        if total_oh // overhangs_needed == 0:
            for rec in current_genes:
                unassigned.append(rec.id)
            break

        survivors, next_phase, pool_sizes = enhanced_assign_n_part_phase(
            current_genes, codon_table, syn_map, n, total_oh
        )

        if not next_phase:
            for idx, pool in enumerate(survivors, start=current_block):
                for (gid, ohs, cuts, seq_mod) in pool["records"]:
                    full = seq_mod
                    fragments = []
                    prev = 0
                    for cut in cuts:
                        fragments.append(full[prev:cut])
                        prev = cut - 4
                    fragments.append(full[prev:])

                    row = {
                        "Block": idx,
                        "Length Distribution": f"Long-{n}part",
                        "Sequence Name": gid,
                        "Full Sequence": full
                    }
                    for i in range(n - 1):
                        row[f"Overhang{i+1}"] = ohs[i] if i < len(ohs) else "N/A"
                    for i in range(n):
                        row[f"DNA Fragment {i+1}"] = fragments[i] if i < len(fragments) else ""
                    rows.append(row)
            break

        threshold = (total_oh // (n - 1)) // 3
        culled_ids = []
        for pool in survivors:
            size = len(pool["records"])
            if size < threshold and size > 1:
                for (gid, _, _, _) in pool["records"]:
                    culled_ids.append(gid)

        culled_recs = [rec for rec in current_genes if rec.id in culled_ids]
        current_genes = culled_recs + next_phase

        for idx, pool in enumerate(survivors, start=current_block):
            size = len(pool["records"])
            if size >= threshold or size == 1:
                for (gid, ohs, cuts, seq_mod) in pool["records"]:
                    full = seq_mod
                    fragments = []
                    prev = 0
                    for cut in cuts:
                        fragments.append(full[prev:cut])
                        prev = cut - 4
                    fragments.append(full[prev:])

                    row = {
                        "Block": idx,
                        "Length Distribution": f"Long-{n}part",
                        "Sequence Name": gid,
                        "Full Sequence": full
                    }
                    for i in range(n - 1):
                        row[f"Overhang{i+1}"] = ohs[i] if i < len(ohs) else "N/A"
                    for i in range(n):
                        row[f"DNA Fragment {i+1}"] = fragments[i] if i < len(fragments) else ""
                    rows.append(row)
                current_block += 1

        n += 1

    # Build DataFrame, sort by "Block", reorder columns, then save
    df_out = pd.DataFrame(rows)
    df_out.sort_values(by="Block", inplace=True)

    # Identify Overhang and Fragment columns, sort them numerically
    oh_cols = sorted([c for c in df_out.columns if c.startswith("Overhang")],
                     key=lambda x: int(x.replace("Overhang", "")))
    frag_cols = sorted([c for c in df_out.columns if c.startswith("DNA Fragment")],
                       key=lambda x: int(x.replace("DNA Fragment ", "")))

    # Base columns before Overhangs/Fragments
    base_cols = ["Block", "Length Distribution", "Sequence Name"]

    # Final column order: base_cols + oh_cols + frag_cols + ["Full Sequence"]
    final_order = base_cols + oh_cols + frag_cols + ["Full Sequence"]
    df_out = df_out[final_order]

    df_out.to_csv(OUTPUT_FILE, index=False)
    write_unassigned_log(unassigned, UNASSIGNED_LOG)

    print("Processing complete.")
    print(f"OPOOL_LENGTH     = {OPOOL_LENGTH}")
    print(f"Vector Overhangs = {VECTOR_OVERHANG_1}, {VECTOR_OVERHANG_2}")
    print(f"Final CSV → {OUTPUT_FILE}")
    if unassigned:
        print(f"Unassigned → {UNASSIGNED_LOG}")
    else:
        print("All genes assigned (2- to n-part).")


if __name__ == "__main__":
    main_pipeline()


## Primer and BsaI site addition

### Variables

In [ ]:
import pandas as pd
from Bio.Seq import Seq

# User‐defined variables
VECTOR_5           = "GCTT"    # 4-nt 5' vector overhang
VECTOR_3           = "AGTG"    # 4-nt 3' vector overhang
TYPEIIS_CUT_SITE   = "GGTCTCA"  # Type IIS enzyme site including specified 'N' (no overhang)

# Filepaths
INPUT_ASSEMBLY_CSV  = "Assigned_Genes.csv"
PRIMER_CSV          = "orthogonal_oligos.csv"
OUTPUT_WITH_PRIMERS = "FULL_INFO.csv"
OUTPUT_FASTA          = "references.fasta"
OUTPUT_FRAGMENTS_CSV  = "oPool_Order_Fragments.csv"


### Script

In [ ]:
def load_primers(path):
    df = pd.read_csv(path, header=None)
    return [(str(r[0]).strip(), str(r[1]).strip().upper()) for _, r in df.iterrows()]

def revcomp(s):
    return str(Seq(s).reverse_complement())

# Load primers and input assembly CSV
primers = load_primers(PRIMER_CSV)
df = pd.read_csv(INPUT_ASSEMBLY_CSV)

# Identify fragment columns and define Type IIS overhangs
frag_cols = [c for c in df.columns if c.startswith("DNA Fragment")]
cut_pref  = TYPEIIS_CUT_SITE
cut_suf   = revcomp(cut_pref)

def process_row(r):
    block = int(r["Block"])
    pi = 2 * (block - 1)
    fwd_name, fwd_seq = primers[pi]
    rev_name, rev_seq = primers[pi + 1]
    rev_seq_suf = revcomp(rev_seq)

    # Determine which fragments are non‐empty
    non_empty = [i for i, col in enumerate(frag_cols) if isinstance(r[col], str) and r[col] != ""]
    new_frags = []
    for i, col in enumerate(frag_cols):
        seq = r[col]
        if not isinstance(seq, str) or seq == "":
            new_frags.append(seq)
            continue

        first = (i == non_empty[0])
        last  = (i == non_empty[-1])
        s = seq
        if first:
            s = VECTOR_5 + s
        if last:
            s = s + VECTOR_3

        s2 = cut_pref + s + cut_suf
        new_frags.append(fwd_seq + s2 + rev_seq_suf)

    # Replace fragment columns with processed sequences
    for col, nf in zip(frag_cols, new_frags):
        r[col] = nf

    # Add primer information to the row
    r["Primer_Forward_Name"] = fwd_name
    r["Primer_Forward_Seq"]  = fwd_seq
    r["Primer_Reverse_Name"] = rev_name
    r["Primer_Reverse_Seq"]  = rev_seq
    return r

# Apply processing, sort by Block
df = df.apply(process_row, axis=1)
df.sort_values(by="Block", inplace=True)

# Define columns for output
oh_cols     = sorted([c for c in df.columns if c.startswith("Overhang")],
                     key=lambda x: int(x.replace("Overhang", "")))
frag_cols   = sorted(frag_cols, key=lambda x: int(x.replace("DNA Fragment ", "")))
base_cols   = ["Block", "Length Distribution", "Sequence Name"]
primer_cols = ["Primer_Forward_Name", "Primer_Forward_Seq", "Primer_Reverse_Name", "Primer_Reverse_Seq"]

final_cols = base_cols + oh_cols + frag_cols + ["Full Sequence"] + primer_cols
for c in final_cols:
    if c not in df.columns:
        df[c] = ""
df = df[final_cols]

# Write CSV with primers included
df.to_csv(OUTPUT_WITH_PRIMERS, index=False)

# Write references.fasta with Block number in the header
with open(OUTPUT_FASTA, "w") as fh:
    for _, row in df.iterrows():
        block = int(row["Block"])
        seq_name = row["Sequence Name"]
        full_seq = row["Full Sequence"]
        header = f"Block_{block}_{seq_name}"
        fh.write(f">{header}\n{full_seq}\n")

# Write fragments.csv (Gene_Fragment, FragmentSequence)
rows = []
for _, row in df.iterrows():
    seq_name = row["Sequence Name"]
    for i, col in enumerate(frag_cols, start=1):
        frag = row[col]
        if isinstance(frag, str) and frag != "":
            rows.append({
                "Gene_Fragment": f"{seq_name}_Fragment{i}",
                "Sequence": frag
            })

frag_df = pd.DataFrame(rows)
frag_df.to_csv(OUTPUT_FRAGMENTS_CSV, index=False)

print(f"Written to '{OUTPUT_WITH_PRIMERS}', '{OUTPUT_FASTA}', and '{OUTPUT_FRAGMENTS_CSV}'.")
